# Lekcija 38: Prijenos učenja

Treniranje dobrog CNN-a od početka (lekcije 34–35) zahtijevalo je stotine označenih primjera čak i za demonstracijski zadatak. Stvarnim ciljnim zadacima često nedostaju podaci: tek nekoliko označenih medicinskih snimki ili nova kategorija proizvoda s 20 fotografija. **Prijenos učenja** to zaobilazi ponovnom uporabom mreže već trenirane na *drugom* zadatku bogatom podacima, uz pretpostavku da su značajke ranih slojeva (rubovi, područja, jednostavne teksture) korisne za gotovo svaki vizualni zadatak, a ne samo za onaj na kojem su izvorno učene. Ova lekcija to pokazuje na stvarnim fotografijama, koristeći podskup CIFAR-10 (<a href="../references.html#krizhevsky-2009-cifar">Krizhevsky, 2009.</a>).

In [ ]:
import copy
import pickle
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Izvorni i ciljni zadatak

Postavimo dva povezana, ali različita zadatka, oba iz CIFAR-10. **Izvorni zadatak** ima mnogo podataka: razlikovanje aviona od automobila, po 300 slika za treniranje. **Ciljni zadatak** onaj je koji nas doista zanima i namjerno ima malo podataka: razlikovanje mačaka od kamiona — dviju klasa koje izvorni zadatak nikada nije vidio — sa samo **15 slika za treniranje po klasi**.

In [ ]:
CIFAR_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
CACHE_ROOT = Path.home() / '.cache' / 'cvintro'
CACHE_DIR = CACHE_ROOT / 'cifar-10-batches-py'

def ensure_cifar10():
    if CACHE_DIR.exists():
        return
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = CACHE_ROOT / 'cifar-10-python.tar.gz'
    if not archive_path.exists():
        print('Downloading CIFAR-10 (~163 MB, one-time, cached under ~/.cache/cvintro)...')
        urllib.request.urlretrieve(CIFAR_URL, archive_path)
    print('Extracting...')
    with tarfile.open(archive_path) as tar:
        tar.extractall(CACHE_ROOT)

def load_cifar_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    imgs = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.float32) / 255.0
    labels = np.array(d[b'labels'], dtype=np.int64)
    return imgs, labels

ensure_cifar10()

CIFAR_LABELS = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
CID = {name: CIFAR_LABELS.index(name) for name in CIFAR_LABELS}

train_imgs, train_labels = [], []
for i in range(1, 6):
    imgs, labels = load_cifar_batch(CACHE_DIR / f'data_batch_{i}')
    train_imgs.append(imgs); train_labels.append(labels)
train_imgs, train_labels = np.concatenate(train_imgs), np.concatenate(train_labels)
test_imgs, test_labels = load_cifar_batch(CACHE_DIR / 'test_batch')

def take(imgs, labels, name, n, rng_local):
    idx = np.where(labels == CID[name])[0]
    idx = rng_local.permutation(idx)[:n]
    return imgs[idx]

data_rng = np.random.default_rng(3)

# source task: airplane vs. automobile, plenty of data
src_a = take(train_imgs, train_labels, 'airplane', 300, data_rng)
src_b = take(train_imgs, train_labels, 'automobile', 300, data_rng)
X_src = np.concatenate([src_a, src_b])
y_src = np.concatenate([np.zeros(300), np.ones(300)]).astype(np.int64)
perm = data_rng.permutation(len(X_src)); X_src, y_src = X_src[perm], y_src[perm]

# target task: cat vs. truck, neither ever seen by the source task, deliberately starved
X_tgt_train = np.concatenate([take(train_imgs, train_labels, 'cat', 15, data_rng),
                               take(train_imgs, train_labels, 'truck', 15, data_rng)])
y_tgt_train = np.array([0] * 15 + [1] * 15, dtype=np.int64)
X_tgt_test = np.concatenate([take(test_imgs, test_labels, 'cat', 150, data_rng),
                              take(test_imgs, test_labels, 'truck', 150, data_rng)])
y_tgt_test = np.array([0] * 150 + [1] * 150, dtype=np.int64)

fig, axes = plt.subplots(2, 6, figsize=(11, 4))
for ax, im in zip(axes[0], list(src_a[:3]) + list(src_b[:3])):
    ax.imshow(im); ax.axis('off')
axes[0, 0].set_ylabel('source', rotation=0, labelpad=25)
for ax, im in zip(axes[1], list(X_tgt_train[:3]) + list(X_tgt_train[15:18])):
    ax.imshow(im); ax.axis('off')
axes[1, 0].set_ylabel('target', rotation=0, labelpad=25)
fig.suptitle('Source task (airplane vs. automobile, top) vs. target task (cat vs. truck, bottom)', y=1.02)
plt.show()

## Tri strategije

1. **Od početka** — trenirati novi CNN samo na 30 ciljnih slika. To je početni rezultat za usporedbu: bez ikakvog prijenosa.
2. **Zamrznuta osnovna mreža (izdvajanje značajki)** — prethodno trenirati osnovnu mrežu CNN-a na izvornom zadatku, zatim potpuno zamrznuti njezine težine i trenirati samo novi linearni klasifikator nad značajkama koje stvara za ciljne slike.
3. **Fino podešavanje** — početi od iste prethodno trenirane osnovne mreže, ali nastaviti ažurirati i nju na ciljnim podacima, uz *mnogo manju* stopu učenja za osnovnu mrežu nego za novi klasifikacijski završni sloj (osnovna mreža već kodira korisnu strukturu; velike promjene iz samo 30 primjera uništile bi je).

In [ ]:
class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 3, padding=1), nn.ReLU(),
            nn.AdaptiveMaxPool2d(1),
        )

    def forward(self, x):
        return self.conv(x).flatten(1)

class Classifier(nn.Module):
    def __init__(self, backbone, freeze_backbone):
        super().__init__()
        self.backbone = backbone
        self.freeze_backbone = freeze_backbone
        self.fc = nn.Linear(32, 2)

    def forward(self, x):
        feat = self.backbone(x)
        if self.freeze_backbone:
            feat = feat.detach()  # freeze backbone so no gradient flows; below, ResNet freezes via requires_grad=False instead of detach() -- both stop gradients, but the latter is cheaper when the backbone is never going to be unfrozen later
        return self.fc(feat)

def train_source(seed, epochs=300, lr=0.001):
    torch.manual_seed(seed)  # seed before constructing the model (Lesson 34/35)
    backbone = Backbone()
    fc = nn.Linear(32, 2)
    opt = torch.optim.Adam(list(backbone.parameters()) + list(fc.parameters()), lr=lr)
    Xt = torch.tensor(X_src).permute(0, 3, 1, 2); yt = torch.tensor(y_src)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.cross_entropy(fc(backbone(Xt)), yt)
        loss.backward()
        opt.step()
    return backbone

def train_target(backbone, freeze_backbone, seed, epochs=200, lr=0.01, backbone_lr=None):
    torch.manual_seed(seed)
    model = Classifier(backbone, freeze_backbone)
    if freeze_backbone:
        opt = torch.optim.Adam(model.fc.parameters(), lr=lr)
    elif backbone_lr is not None:
        opt = torch.optim.Adam([
            {'params': model.backbone.parameters(), 'lr': backbone_lr},
            {'params': model.fc.parameters(), 'lr': lr},
        ])
    else:
        opt = torch.optim.Adam(model.parameters(), lr=lr)
    Xtr = torch.tensor(X_tgt_train).permute(0, 3, 1, 2); ytr = torch.tensor(y_tgt_train)
    Xte = torch.tensor(X_tgt_test).permute(0, 3, 1, 2); yte = torch.tensor(y_tgt_test)
    for _ in range(epochs):
        opt.zero_grad()
        loss = F.cross_entropy(model(Xtr), ytr)
        loss.backward()
        opt.step()
    with torch.no_grad():
        return (model(Xte).argmax(1) == yte).float().mean().item()

In [ ]:
pretrained = train_source(seed=0)  # pretrain once; the point of transfer learning is reusing this

scratch_accs, frozen_accs, finetune_accs = [], [], []
for seed in range(8):
    scratch_acc = train_target(Backbone(), freeze_backbone=False, seed=seed)
    frozen_acc = train_target(copy.deepcopy(pretrained), freeze_backbone=True, seed=seed)
    finetune_acc = train_target(copy.deepcopy(pretrained), freeze_backbone=False, seed=seed, backbone_lr=0.0003)
    scratch_accs.append(scratch_acc); frozen_accs.append(frozen_acc); finetune_accs.append(finetune_acc)

print(f'{"strategy":>20} {"mean test acc":>16} {"std":>8}')
print(f'{"from scratch":>20} {np.mean(scratch_accs):>15.1%} {np.std(scratch_accs):>8.1%}')
print(f'{"frozen backbone":>20} {np.mean(frozen_accs):>15.1%} {np.std(frozen_accs):>8.1%}')
print(f'{"fine-tuned":>20} {np.mean(finetune_accs):>15.1%} {np.std(finetune_accs):>8.1%}')
print(f'\n(averaged over {len(scratch_accs)} random seeds, each reusing the same 30 target training images)')

Obje strategije prijenosa jasno nadmašuju treniranje od početka, a ujedno su i znatno *dosljednije* (manja standardna devijacija). Iako izvorni zadatak nikada nije vidio mačku ni kamion, naučio je opće značajke niske razine (rubove, obojena područja, teksture) koje su se prenijele na novi zadatak.

Primijetite da je fino podešavanje koristilo stopu učenja osnovne mreže približno 30 puta manju od stope klasifikacijskog završnog sloja. Uz samo 30 ciljnih primjera veće ažuriranje osnovne mreže dovelo bi do prenaučenosti na tih 30 slika i odbacivanja svega korisnog naučenog na izvornom zadatku sa 600 slika. To je isti način neuspjeha katastrofalnog zaboravljanja kao kod krivulje prenaučenosti iz lekcije 35, samo primijenjen na mrežu koja je u početku već nešto znala.

Još jedan detalj: `pretrained` se trenira jednom (`seed=0`) i ponovno koristi kroz svih osam inicijalizacija u gornjoj petlji. Samo treniranje na ciljnom zadatku (inicijalizacija klasifikatora, putanja finog podešavanja) varira s `seed`, pa gore prikazane standardne devijacije mjere samo varijabilnost ciljnog treniranja, a ne šum prethodnog treniranja na izvornom zadatku.

## U praksi: prethodno trenirane osnovne mreže u jednom retku

Gornji `Backbone` treniran je od početka na demonstracijskom zadatku radi ilustracije, ali stvarni prijenos učenja koristi postojeće mreže dostupne za ponovnu uporabu. `torchvision` sadrži arhitekture već prethodno trenirane na cijelom skupu ImageNet (lekcija 37) — 1.2 milijuna slika u 1000 klasa. Učitavanje takve mreže, spremne za uporabu kao zamrznutog izdvajatelja značajki, jedan je redak koda:

In [ ]:
import torchvision

resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1)
n_total = sum(p.numel() for p in resnet.parameters())

resnet.fc = nn.Identity()  # strip the original 1000-class ImageNet head, keep the 512-d features underneath
resnet.eval()
for p in resnet.parameters():
    p.requires_grad = False  # frozen, feature-extraction style — the same strategy as "frozen backbone" above

print(f'total ResNet-18 parameters: {n_total:,}  (all pretrained on ImageNet, none of them touched here)')

ResNet-18 treniran je na slikama 224x224 normaliziranima srednjom vrijednošću i standardnom devijacijom svakog kanala ImageNeta — a ne običnim skaliranjem piksela na `[0, 1]` korištenim drugdje u ovoj lekciji — pa ciljne slike treba promijeniti na odgovarajuću veličinu i ponovno normalizirati prije prolaska kroz mrežu. Jednom izdvojite 512-dimenzijski vektor značajki za svaku ciljnu sliku (osnovna mreža je zamrznuta, pa to treba učiniti samo jednom, a ne pri svakom koraku treniranja), a zatim nad njima trenirajte mali linearni završni sloj — upravo kao u prethodnoj strategiji zamrznute osnovne mreže, samo uz znatno drukčiju osnovnu mrežu.

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

def extract_features(X):
    x = torch.tensor(X).permute(0, 3, 1, 2)
    x = F.interpolate(x, size=224, mode='bilinear', align_corners=False)
    x = (x - IMAGENET_MEAN) / IMAGENET_STD
    with torch.no_grad():
        return torch.cat([resnet(x[i:i + 32]) for i in range(0, len(x), 32)])

feat_train = extract_features(X_tgt_train)
feat_test = extract_features(X_tgt_test)
ytr_t = torch.tensor(y_tgt_train); yte_t = torch.tensor(y_tgt_test)

resnet_accs = []
for seed in range(8):
    torch.manual_seed(seed)
    head = nn.Linear(feat_train.shape[1], 2)
    opt = torch.optim.Adam(head.parameters(), lr=0.01)
    for _ in range(200):
        opt.zero_grad()
        loss = F.cross_entropy(head(feat_train), ytr_t)
        loss.backward()
        opt.step()
    with torch.no_grad():
        resnet_accs.append((head(feat_test).argmax(1) == yte_t).float().mean().item())

print(f'frozen real ImageNet ResNet-18: mean test acc = {np.mean(resnet_accs):.1%}  (+/- {np.std(resnet_accs):.1%}, 8 seeds)')
print(f'(toy frozen backbone above got {np.mean(frozen_accs):.1%})')

Zamrznute značajke ImageNeta podižu točnost znatno iznad obiju prethodnih demonstracijskih strategija, koristeći samo novi linearni sloj treniran na istih 30 ciljnih slika — bez ikakvog treniranja osnovne mreže.

**Ograničenje ove usporedbe:** dio te razlike dolazi iz nepravedne prednosti, a ne samo bogatijeg prethodnog treniranja. ImageNetovih 1000 klasa već uključuje fotografije mačaka i nekoliko vrsta vozila, za razliku od demonstracijskog izvornog zadatka, koji je vidio samo avione i automobile. Zato ovo nije potpuno kontrolirana usporedba isključivo „velikog općeg prethodnog treniranja” i „malog prethodnog treniranja na 2 klase”; dio prednosti ResNeta-18 dolazi iz toga što je tijekom prethodnog treniranja izravno vidio ciljne klase.

Ipak, kvalitativna poanta vrijedi i upravo je ona razlog zbog kojeg su osnovne mreže prethodno trenirane na ImageNetu uobičajeno polazište u praksi: mreža trenirana na 1.2 milijuna slika uči mnogo bogatije i šire prenosive značajke nego što bi ikada mogla mreža trenirana na nekoliko tisuća slika dviju kategorija — a upravo su za to izgrađene arhitekture iz lekcije 37 i mnogo veći skupovi podataka na kojima su trenirane.

### Zadaci

1. Isprobajte `backbone_lr=0.01` (odnosno bez razlike u stopi učenja osnovne mreže i završnog sloja) u pozivu finog podešavanja. Poboljšava li se ili pogoršava točnost fino podešenog modela i odgovara li to gornjem objašnjenju katastrofalnog zaboravljanja?
2. Pokušajte zamrznuti *većinu* osnovne mreže, ali fino podešavati samo njezin posljednji konvolucijski sloj (savjet: postavite `requires_grad = False` samo na parametre prvog `Conv2d`). Gdje je taj rezultat u odnosu na potpuno zamrznutu i potpuno fino podešenu mrežu?
3. Promijenite ciljne klase s mačke/kamiona na psa/automobil — ciljni par čija se *domena* (vozila) još više preklapa s izvornim zadatkom nego kod mačke/kamiona. Poboljšava li se točnost zamrznute osnovne mreže dodatno i odgovara li to intuiciji da prijenos najbolje radi kada izvorni i ciljni zadatak dijele više temeljne vizualne strukture?
4. `resnet.fc = nn.Identity()` ostavlja sve konvolucijske slojeve ResNeta-18 zamrznutima. Pokušajte odmrznuti samo njegovu posljednju rezidualnu fazu (`resnet.layer4`) i fino je podešavati na 30 ciljnih slika uz malu stopu učenja, kao što je prethodno fino podešavana demonstracijska osnovna mreža. Nadmašuje li rezultat potpuno zamrznutog ResNeta-18 ili je 30 slika premalo za pouzdano ažuriranje čak i malog dijela od 11 milijuna parametara?